# 第31章：模型验证

先按标签区间交集做净化与embargo，再用反例区分行数gap与真实时间约束。随后在20项资产、500日的单种子世界中比较泄漏、1-NN、K折、前推和嵌套拟合；这不是旧五种子完整协议的回放。代码新写一张purge图。label_end不自动等于真实可得时间，French三月标签分支默认跳过。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch31_model_validation_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 半开标签区间、purge与embargo

区间为 $(t,t+5]$，测试6/7/8，(1,6]与(6,11]接触而不交叠；净化及m=3后留[0,1]。区间终点不保证标签结算/发布/可读取；实际训练资格须另核a≤fit_time。本格新保存purge示意图。

In [ ]:
import numpy as np
import pandas as pd
from quant_course.validation import (
    check_label_intervals,
    intervals_overlap,
    purge,
    embargo,
)

# 观测日期 t = 0..14，标签跨度 h = 5 => 标签区间 I_t = (t, t + 5]
t = np.arange(15, dtype=float)
h = 5
e = t + h

test_idx = np.array([6, 7, 8])
cand_train = np.array([0, 1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 14])

# 1. 运行 Purge：剔除与测试集标签区间有交集的训练样本
purged_train = purge(cand_train, test_idx, t, e)
removed_purge = np.setdiff1d(cand_train, purged_train)

print("测试样本索引:", test_idx.tolist())
print("候选训练集:", cand_train.tolist())
print("Purge 剔除的样本:", removed_purge.tolist())
print("Purge 保留的样本:", purged_train.tolist())

# 2. 验证接触非重叠：样本 1 (1, 6] 与测试样本 6 (6, 11] 在 t=6 接触
assert 1 in purged_train
assert not intervals_overlap(t[1], e[1], t[6], e[6])
print("接触非重叠验证成功：样本 1 标签区间 (1, 6] 与测试样本 6 区间 (6, 11] 仅在 6 接触，未重叠，样本 1 得以保留。")

# 3. 运行 Embargo：特征回看窗口 m = 3，仅对测试集之后的训练样本进行过滤
final_train = embargo(purged_train, test_idx, t, e, m=3)
removed_embargo = np.setdiff1d(purged_train, final_train)

print("Embargo 剔除的样本:", removed_embargo.tolist())
print("最终保留的训练样本集:", final_train.tolist())
assert np.array_equal(final_train, [0, 1])

# 4. 可视化手算例标签区间、Purge 与 Embargo（本次图写到 RUN_DIR）
import matplotlib.pyplot as plt
ch31 = load_code_module("ch31_model_validation.py")
h_data = ch31.run_hand_example()
fig, ax = ch31.plot_hand_example(h_data)
fig.savefig(RUN_DIR / "ch31_purge_embargo.png", dpi=150)
plt.show()

## 2. 样本计数（gap）与时间区间（purge）的等价与失效边界

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# 1. 单资产规则日历序列：gap = h - 1 = 4 恰好等价于 purge
t_single = np.arange(15, dtype=float)
cand_train_single = np.arange(6)
test_single = np.array([6, 7, 8])

purged_single = purge(cand_train_single, test_single, t_single, t_single + 5)
tss = TimeSeriesSplit(n_splits=2, test_size=3, gap=4)
train_tss_single, test_tss_single = next(tss.split(np.arange(12)))

print("单资产序列：")
print("  Purge 保留的训练集:", purged_single.tolist())
print("  TimeSeriesSplit(gap=4) 保留的训练集:", train_tss_single.tolist())
assert np.array_equal(purged_single, train_tss_single)
print("  => 单资产规则日历下，gap = h - 1 完全等价于 Purge！")

# 2. 三资产面板：按 (date, asset) 排序，gap = 4 样本行出现严重前视泄漏
dates_panel = np.repeat(np.arange(15), 3)
e_panel = dates_panel + 5
test_panel_idx = np.arange(18, 27)  # 日期 6..8 共 9 行
cand_train_panel = np.arange(18)    # 日期 0..5 共 18 行

purged_panel = purge(cand_train_panel, test_panel_idx, dates_panel, e_panel)
tss_panel = TimeSeriesSplit(n_splits=2, test_size=9, gap=4)
train_tss_panel, _ = next(tss_panel.split(np.arange(36)))

overlapping_rows = [idx for idx in train_tss_panel if dates_panel[idx] in {2, 3, 4, 5}]

print("\n三资产面板：")
print("  Purge 规范保留的训练行数:", len(purged_panel), "行 (日期 0..1)")
print("  TimeSeriesSplit(gap=4) 保留的训练行数:", len(train_tss_panel), "行")
print("  TimeSeriesSplit 误留在训练集中的重叠行数:", len(overlapping_rows), "行！")
print("  具体残留重叠行索引:", overlapping_rows)
assert len(overlapping_rows) == 8

## 3. 数据泄漏对照：特定合成 DGP 下的观测分数差异

本节比较只描述当前数据生成机制、模型、随机种子和划分。它不估计普遍泄漏幅度，也不构成显著性检验或真实市场表现结论。


In [ ]:
ch31 = load_code_module("ch31_model_validation.py")

# 生成一组已知数据生成过程 (DGP) 的日度合成面板
seed = 20261031
X, y, t_obs, e_obs, dates = ch31.generate_synthetic_panel(seed=seed, n_assets=20, n_dates=500)
print(f"合成面板样本数: {len(y)}, 特征维度: {X.shape[1]}, 日期数: {len(np.unique(dates))}")

# 载入离线运行的 5 种子完整实验结果进行横向复核
results = ch31.run_hand_example()
leakage = ch31.run_leakage_study(seeds=[seed])  # 在 Notebook 中快速运行单种子对照
schemes = leakage["schemes"]

summary_df = pd.DataFrame({
    "方案 (Validation Scheme)": [
        "1. Shuffled K-Fold (随机打散, 全局归一化)",
        "2. Naive TSS (gap=0, 全局归一化)",
        "3. TSS (gap=4, 全局归一化)",
        "4. Purged Walk-Forward (因果净化, 仅训练集归一化)",
    ],
    "样本外 R²": [
        schemes["shuffled_kfold"]["r2_mean"],
        schemes["naive_tss_gap0"]["r2_mean"],
        schemes["tss_gap4_global_scale"]["r2_mean"],
        schemes["purged_walk_forward"]["r2_mean"],
    ],
    "样本外 Rank IC": [
        schemes["shuffled_kfold"]["rank_ic_mean"],
        schemes["naive_tss_gap0"]["rank_ic_mean"],
        schemes["tss_gap4_global_scale"]["rank_ic_mean"],
        schemes["purged_walk_forward"]["rank_ic_mean"],
    ],
    "样本外 MSE": [
        schemes["shuffled_kfold"]["mse_mean"],
        schemes["naive_tss_gap0"]["mse_mean"],
        schemes["tss_gap4_global_scale"]["mse_mean"],
        schemes["purged_walk_forward"]["mse_mean"],
    ],
})

print("\n--- 验证方案消融对比表 ---")
print(summary_df.to_string(index=False))

optimism_r2 = schemes["shuffled_kfold"]["r2_mean"] - schemes["purged_walk_forward"]["r2_mean"]
optimism_ic = schemes["shuffled_kfold"]["rank_ic_mean"] - schemes["purged_walk_forward"]["rank_ic_mean"]
print(f"\n随机打散 K-Fold 与 Purged Walk-Forward 的观测分数差:")
print(f"  R² 差值: {optimism_r2:.4f}")
print(f"  Rank IC 差值: {optimism_ic:.4f}")

# 补充实验：本合成标签机制下的 1-NN 重叠对照
overlap_mem = ch31.run_overlap_memorization_study(seeds=[seed])
print("\n--- 本合成设计下的 1-NN 标签重叠对照 ---")
print(f"  未净化 1-NN 样本外 Rank IC: {overlap_mem['unpurged_rank_ic_mean']:.4f}")
print(f"  Purge 净化 1-NN 样本外 Rank IC: {overlap_mem['purged_rank_ic_mean']:.4f}")
print(f"  本构造下未净化与净化的 Rank IC 差值: {overlap_mem['rank_ic_inflation']:.4f}")

## 4. Purged K-Fold 与 Purged Walk-Forward 的对比

In [ ]:
pkf_res = ch31.run_purged_kfold_vs_walkforward(seed=seed)
print("Purged K-Fold 表现:")
print(f"  R²: {pkf_res['purged_kfold']['r2']:.4f}")
print(f"  Rank IC: {pkf_res['purged_kfold']['rank_ic']:.4f}")
print(f"  保留样本比例: {pkf_res['purged_kfold']['mean_retention_rate']:.4f}")

print("\nPurged Walk-Forward 表现:")
print(f"  R²: {pkf_res['purged_walk_forward']['r2']:.4f}")
print(f"  Rank IC: {pkf_res['purged_walk_forward']['rank_ic']:.4f}")
print(f"  保留样本比例: {pkf_res['purged_walk_forward']['mean_retention_rate']:.4f}")

print("\n结论与经济学解释:")
print(pkf_res["interpretation"])

## 5. 嵌套超参数选择（Nested Selection）与外层评估

In [ ]:
nested_res = ch31.run_nested_selection_study(seed=seed)

print(f"每折内层验证选出的最优超参数 alpha:")
for i, p in enumerate(nested_res["best_params_per_fold"]):
    print(f"  折 {i}: alpha = {p['alpha']}")

print(f"\n嵌套选择外层 MSE: {nested_res['mean_nested_outer_mse']:.6f}")
print(f"直接在外层测试集调参的 MSE: {nested_res['mean_naive_cheating_mse']:.6f}")
print(f"两种评估方式的 MSE 差值: {nested_res['mean_nested_outer_mse'] - nested_res['mean_naive_cheating_mse']:.6f}")

def excerpt(indices):
    return indices[:5] + (["…"] if len(indices) > 10 else []) + (indices[-5:] if len(indices) > 10 else [])

for i, (outer, inner_folds) in enumerate(zip(nested_res["outer_splits_used"], nested_res["inner_splits_used"])):
    train, test = outer["train_idx"], outer["test_idx"]
    print(f"  外层折 {i}: train n={len(train)}, idx={excerpt(train)}; test n={len(test)}, idx={excerpt(test)}")
    print(f"    外层边界: max(e_train)={np.max(e_obs[train]):.0f}, min(t_test)={np.min(t_obs[test]):.0f}")
    for j, inner in enumerate(inner_folds):
        itr, val = inner["train_idx"], inner["val_idx"]
        print(f"    内层折 {j}: train idx={excerpt(itr)}; validation idx={excerpt(val)}")
        print(f"      内层边界: max(e_train)={np.max(e_obs[itr]):.0f}, min(t_validation)={np.min(t_obs[val]):.0f}")


## 6. 真实市场案例：Kenneth French 10 行业月度收益率（1926–2026）

In [ ]:
if RUN_FRENCH:
    from quant_course.french_frozen import load_industries_and_rf
    
    ind_returns, rf, prov = load_industries_and_rf()
    print("数据集来源:", prov["source"])
    print(f"样本区间: {prov['first_month']} 至 {prov['last_month']}, 共 {prov['months']} 个月")
    print(f"资产列: {list(ind_returns.columns)}")
    
    # 运行 3 个月重叠未来标签的滚动验证对比
    french_res = ch31.run_french_market_study()
    print("\n120 个月滚动训练窗、12 个月测试窗对比 (共 88 个评估期):")
    print(f"  朴素 Walk-Forward (无 Purge) - MSE: {french_res['naive_walk_forward']['mse_mean']:.6f}, Rank IC: {french_res['naive_walk_forward']['rank_ic_mean']:.4f}")
    print(f"  因果 Purged Walk-Forward      - MSE: {french_res['purged_walk_forward']['mse_mean']:.6f}, Rank IC: {french_res['purged_walk_forward']['rank_ic_mean']:.4f}")
    print("注意：该修订历史和固定模型下两方案平均 Rank IC 为负；此处未给出均值显著性检验，不据此推断总体无正向预测力，也不构成可交易 Alpha 宣称。")
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")
